# Day 71: Recurrent Neural Networks (RNN) & LSTM

## Why RNNs for Sequences?
Dense layers and CNNs process an entire input at once and have no built-in concept of "order" - shuffling the input features would give a Dense layer identical results. Recurrent Neural Networks (RNNs) are designed for sequential data instead: they process one timestep at a time, and at each step they combine the new input with a "hidden state" that summarizes everything seen so far, effectively giving the network a running memory of the sequence. This makes RNNs a natural fit for anything where order matters and earlier values affect the meaning of later ones - text (word order changes meaning), time series (yesterday's value affects today's), and audio. The code below builds a small synthetic sequence dataset (200 sequences, each 10 timesteps of a single feature) with a label based on the sum across the sequence, which we'll use to compare a SimpleRNN against an LSTM.</cell id="cell-1">


In [ ]:
# pip install tensorflow
import numpy as np

np.random.seed(0)  # reproducible random data
X = np.random.randn(200, 10, 1)  # 200 sequences, each 10 timesteps long, 1 feature per timestep (shape RNN/LSTM layers expect: samples, timesteps, features)
y = (X.sum(axis=1).ravel() > 0).astype(int)  # label: 1 if the values across the sequence sum to a positive number

## Simple RNN Layer
`SimpleRNN` is the most basic recurrent layer: at each timestep it takes the current input and the previous hidden state, combines them with a shared set of weights, and produces a new hidden state - the same weights are reused at every timestep, similar in spirit to how a CNN reuses the same filter across positions. `input_shape=(10, 1)` tells the layer to expect sequences of 10 timesteps with 1 feature each, matching the data built above. In theory this lets the network use information from early in the sequence when making decisions at the end, but in practice plain RNNs struggle with long sequences because of the vanishing gradient problem - as the error signal is backpropagated through many timesteps, it tends to shrink toward zero, so the network effectively "forgets" anything far in the past. This is exactly the limitation LSTM was designed to fix, which we compare against next.</cell id="cell-3">


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

rnn_model = keras.Sequential([
    layers.SimpleRNN(16, input_shape=(10, 1)),  # 16 hidden units; processes the sequence one timestep at a time
    layers.Dense(1, activation='sigmoid')  # binary output from the final hidden state
])
rnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
rnn_model.fit(X, y, epochs=10, verbose=0)
print('Final accuracy:', rnn_model.evaluate(X, y, verbose=0)[1])

## LSTM Layer
LSTM (Long Short-Term Memory) is a more sophisticated recurrent layer that adds a separate "cell state" (a long-term memory channel) alongside the hidden state, controlled by three learned gates: an input gate (what new information to add), a forget gate (what old information to discard), and an output gate (what to expose as the hidden state). These gates let the network learn to preserve important information over many timesteps and discard irrelevant information, which largely solves the vanishing gradient problem that limits plain RNNs. This makes LSTM (and its close relative GRU) far better at capturing long-range dependencies - patterns where something far earlier in a sequence still matters much later - which is why LSTM/GRU became the standard recurrent layer for text and time series before transformers took over many of those tasks. The code here mirrors the SimpleRNN example exactly except for swapping the layer, so you can directly compare the two.</cell id="cell-5">


In [ ]:
lstm_model = keras.Sequential([
    layers.LSTM(16, input_shape=(10, 1)),  # 16 hidden units; gated memory helps retain longer-range patterns than SimpleRNN
    layers.Dense(1, activation='sigmoid')
])
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.fit(X, y, epochs=10, verbose=0)
print('Final accuracy:', lstm_model.evaluate(X, y, verbose=0)[1])

## When to Use RNN/LSTM
The common thread across good use cases for RNN/LSTM is: data that comes in an order, where that order carries meaning and where context from earlier in the sequence should influence how you interpret what comes later. Text is the classic example (word order and preceding words shape meaning), but the same reasoning applies to stock prices and other time series (past values inform the trend), sensor/IoT readings, and audio or speech signals. If your data doesn't have this sequential structure - e.g. a table of independent customer records - a Dense network or tree-based model is usually a better and simpler fit than an RNN/LSTM. Note that for many modern text tasks, transformer-based architectures (like BERT or GPT) have largely replaced RNN/LSTM because they handle long-range context more effectively and train faster in parallel, but RNN/LSTM remain a solid, lightweight choice for smaller sequence problems and are still important for understanding how sequence modeling works.</cell id="cell-7">


In [ ]:
print('Use RNN/LSTM for: text, time series, speech, any sequential/ordered data')